In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os

# ==========================================================
# CONFIGURATION: Modify these variables to run your 15 tasks
# ==========================================================
ALGORITHM = 'SURF'      # Options: 'SIFT', 'SURF', 'ORB', 'AKAZE'
TASK_MODE = 'base'    # Options: 'base', 'rotate', 'scale', 'blur', 'bright', 'dark', 'noise'
IMAGE_FILE = "myyra.jpg" # Ensure this file is in the same folder


img2 = cv2.imread("myyra30.jpg")
#img2= cv2.imread("kaukaa.jpg")
#img2 = cv2.imread("keinussa.jpg")
#img2 = cv2.imread("puolet.jpg")

for kuva in [img2, img3, img5, img5]:
    cv2.resize(kuva, (1050, 1610))

"""
rows, cols, ch = img.shape
src_pts = np.float32([[0, 0], [cols, 0], [0, rows], [cols, rows]])
dst_pts = np.float32([[0, 0], [cols, 0], [0, rows*0.2], [cols, rows*0.8]]) """


def apply_transformation(img, mode):
    """
    Generates the transformed image based on the selected Task Mode.
    Students can modify the parameters here (angles, scale factors, etc.)
    """
    if mode == 'base':
        return img.copy()
    elif mode == 'rotate':
        # Task 1 & 2: Change angle to 43 or 90
        (h, w) = img.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 90, 1.0) 
        return cv2.warpAffine(img, M, (w, h))
    elif mode == 'tilt':
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        return cv2.warpPerspective(img, M, (cols, rows))
        
    elif mode == 'flip':
        return cv2.flip(img, 0)
    elif mode == 'scale':
        # Task 3 & 4: Change fx/fy to 0.25 or 2.0
        return cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_LINEAR)
    elif mode == 'blur':
        # Task 8 & 9: Increase (15, 15) for more blur
        return cv2.GaussianBlur(img, (29, 29), 0)

    elif mode == 'smear':
        psf = np.zeros((30, 30, 3))
        psf = cv2.ellipse(psf, 
                  (25, 25), # center
                  (20, 1), # axes -- 22 for blur length, 0 for thin PSF 
                  30, # angle of motion in degrees
                  0, 360, # ful ellipse, not an arc
                  (1, 1, 1), # white color
                  thickness=-1) # filled

        psf /= psf[:,:,0].sum() # normalize by sum of one channel 
                        # since channels are processed independently

        return cv2.filter2D(img, -1, psf)
        
    elif mode == 'bright':
        # Task 7: Brighten
        return cv2.convertScaleAbs(img, alpha=1.5, beta=50)
    elif mode == 'dark':
        # Task 6: Darken
        return cv2.convertScaleAbs(img, alpha=0.5, beta=-50)
    elif mode == 'noise':
        # Task 10: Simulate JPEG artifacts/noise
        noise = np.random.randint(0, 100, img.shape, dtype='uint8')
        return cv2.add(img, noise)
        
    elif mode == 'save':
        cv2.imwrite('tree_1.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 1])
        return cv2.imread('tree_1.jpg')

    elif mode == 'color':
        rows, cols = img.shape[:2]
        # Create warm (yellow) and cool (blue) overlays
        warm_bg = np.full((rows, cols, 3), (0, 180, 255), dtype=np.uint8)
        
        # Blend overlays
        warm = cv2.addWeighted(img, 0.7, warm_bg, 0.3, 0)
        return warm
    
    return img
    
    
    

def get_detector(name):
    """Initializes the algorithm and the correct distance norm."""
    if name == 'SIFT':
        return cv2.SIFT_create(), cv2.NORM_L2
    elif name == 'SURF':
        # Note: SURF might require 'opencv-contrib-python'
        try:
            return cv2.xfeatures2d.SURF_create(hessianThreshold=400), cv2.NORM_L2
        except:
            print("SURF not found. Ensure opencv-contrib-python is installed.")
            return None, None
    elif name == 'ORB':
        return cv2.ORB_create(nfeatures=1000), cv2.NORM_HAMMING
    elif name == 'AKAZE':
        return cv2.AKAZE_create(), cv2.NORM_HAMMING
    return None, None
    

def main():
    # 1. Load Original Image
    
    original = cv2.imread("myyra.jpg")
    
    #cool_bg = np.full((rows, cols, 3), (255, 140, 0), dtype=np.uint8)
    #original = cv2.addWeighted(original, 0.7, cool_bg, 0.3, 0)
    
    if original is None:
        print(f"Error: {IMAGE_FILE} not found!")
        return

    # 2. Apply Transformation for the second image
    transformed = apply_transformation(img2, TASK_MODE)

    # 3. Setup Algorithm
    detector, norm_type = get_detector(ALGORITHM)
    if detector is None: return

    # 4. Detect & Compute (Timing Task 15)
    t0 = time.time()
    kp1, des1 = detector.detectAndCompute(original, None)
    kp2, des2 = detector.detectAndCompute(transformed, None)
    t1 = time.time()

    print(f"\n--- Result for {ALGORITHM} on Task: {TASK_MODE} ---")
    print(f"Time Taken: {(t1-t0)*1000:.2f} ms")
    print(f"Keypoints Image 1: {len(kp1)} | Image 2: {len(kp2)}")

    # 5. Matching logic
    bf = cv2.BFMatcher(norm_type, crossCheck=True)
    if des1 is not None and des2 is not None:
        matches = bf.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)

        # Draw top 50 matches
        out_img = cv2.drawMatches(original, kp1, transformed, kp2, matches[:50], None, 
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

        # 6. Visualization
        plt.figure(figsize=(15, 7))
        plt.title(f"Alg: {ALGORITHM} | Task: {TASK_MODE} | Matches: {len(matches)}")
        plt.imshow(cv2.cvtColor(out_img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
    else:
        print("No matches found.")

if __name__ == "__main__":
    main()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Henna\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Henna\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\Henna\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\Henna\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import